# ETL III: Normalize

Follow along with the lesson page on the course site. Each heading below matches a section of the lesson: write that section's code in the cell beneath it, and add more cells whenever you need them.

## Setup

In [ ]:
import polars as pl

flat = pl.read_csv(
    "data/courses_flat.csv",
    schema_overrides={
        "course_number": pl.String,
        "room": pl.String,
        "start_time": pl.Time,
        "end_time": pl.Time,
    },
)

print(flat.shape)
flat.head()

## Step 1: Build the Lookup Tables

## Step 2: Build `courses`, `sections`, and `meetings`

## Step 3: Build `instructors` With a Generated Key

## Step 4: Build the Bridge Table

**Checkpoint 1: Collect the tables**

Your notebook already has this code. Run it, then check that your output matches what's below.

You should see:

```
terms                      1 rows
subjects                  46 rows
buildings                  9 rows
courses                  426 rows
sections                 676 rows
meetings                 745 rows
instructors              228 rows
section_instructors      691 rows
```

If a count doesn't match, go back to the step that built that table: Step 1 for `terms`, `subjects`, and `buildings`, Step 2 for `courses`, `sections`, and `meetings`, Step 3 for `instructors`, and Step 4 for `section_instructors`. If you get a `NameError`, a table is missing: run every cell above this one, in order.

In [ ]:
tables = {
    "terms": terms,
    "subjects": subjects,
    "buildings": buildings,
    "courses": courses,
    "sections": sections,
    "meetings": meetings,
    "instructors": instructors,
    "section_instructors": section_instructors,
}

for name, df in tables.items():
    print(f"{name:<22}{len(df):>6,} rows")

## Step 5: Verify Primary Keys

**Checkpoint 2: Unique primary keys**

Your notebook already has this code. Run it, then check that your output matches what's below.

You should see:

```
terms                 0 duplicate keys
subjects              0 duplicate keys
buildings             0 duplicate keys
courses               0 duplicate keys
sections              0 duplicate keys
meetings              0 duplicate keys
instructors           0 duplicate keys
section_instructors   0 duplicate keys
```

If a table has duplicate keys, its `select()` most likely includes a column that doesn't belong in that table, or it's missing `.unique()`.

In [ ]:
primary_keys = {
    "terms": ["term_code"],
    "subjects": ["subject_code"],
    "buildings": ["building_code"],
    "courses": ["subject_code", "course_number"],
    "sections": ["term_code", "crn"],
    "meetings": ["term_code", "crn", "meeting_number"],
    "instructors": ["instructor_id"],
    "section_instructors": ["term_code", "crn", "instructor_id"],
}

for name, key in primary_keys.items():
    duplicates = tables[name].select(key).is_duplicated().sum()
    print(f"{name:<22}{duplicates} duplicate keys")

## Step 6: Verify Referential Integrity

**Checkpoint 3: No orphans**

Your notebook already has this code. Run it, then check that your output matches what's below.

You should see:

```
courses → subjects: 0 orphans
sections → terms: 0 orphans
sections → courses: 0 orphans
meetings → sections: 0 orphans
meetings → buildings: 0 orphans
section_instructors → sections: 0 orphans
section_instructors → instructors: 0 orphans
```

An orphan means the parent table is missing rows. Check that the parent was built from all of `flat`, and that it wasn't filtered along the way.

In [ ]:
def orphans(child, parent, child_cols, parent_cols):
    """Return the distinct foreign key values in `child` with no match in `parent`."""
    return (
        child.select(child_cols)
             .drop_nulls()
             .unique()
             .join(parent, left_on=child_cols, right_on=parent_cols, how="anti")
    )

foreign_keys = [
    ("courses", "subjects", ["subject_code"], ["subject_code"]),
    ("sections", "terms", ["term_code"], ["term_code"]),
    ("sections", "courses", ["subject_code", "course_number"], ["subject_code", "course_number"]),
    ("meetings", "sections", ["term_code", "crn"], ["term_code", "crn"]),
    ("meetings", "buildings", ["building_code"], ["building_code"]),
    ("section_instructors", "sections", ["term_code", "crn"], ["term_code", "crn"]),
    ("section_instructors", "instructors", ["instructor_id"], ["instructor_id"]),
]

for child, parent, child_cols, parent_cols in foreign_keys:
    n = len(orphans(tables[child], tables[parent], child_cols, parent_cols))
    print(f"{child} → {parent}: {n} orphans")

## Step 7: Verify Nothing Was Lost

**Checkpoint 4: Nothing was lost**

Your notebook already has this code. Run it, then check that your output matches what's below.

You should see:

```
770 rows rebuilt, 770 in the flat table
Identical: True
```

If the row counts differ, a table is missing rows or has extra ones, and Checkpoints 1 and 2 will usually point to which. If the counts match but you see `Identical: False`, a column was changed along the way, so compare a few rows of `rebuilt` and `flat` to find it.

In [ ]:
rebuilt = (
    sections
    .join(terms, on="term_code", how="left")
    .join(subjects, on="subject_code", how="left")
    .join(meetings, on=["term_code", "crn"], how="left")
    .join(buildings, on="building_code", how="left")
    .join(section_instructors, on=["term_code", "crn"], how="left")
    .join(instructors, on="instructor_id", how="left")
    .select(flat.columns)
)

print(f"{len(rebuilt):,} rows rebuilt, {len(flat):,} in the flat table")
print("Identical:", rebuilt.sort(flat.columns).equals(flat.sort(flat.columns)))

## Step 8: Save the Tables

**Checkpoint 5: Save the tables**

Your notebook already has this code. Run it, then check that your output matches what's below.

You should see:

```
['buildings.csv', 'courses.csv', 'instructors.csv', 'meetings.csv', 'section_instructors.csv', 'sections.csv', 'subjects.csv', 'terms.csv']
```

You should also see a new `tables` folder inside `data` in VS Code's Explorer, holding the same eight files.

In [ ]:
import pathlib

table_dir = pathlib.Path("data/tables")
table_dir.mkdir(parents=True, exist_ok=True)

for name, df in tables.items():
    df.write_csv(table_dir / f"{name}.csv", time_format="%H:%M:%S")

print(sorted(path.name for path in table_dir.glob("*.csv")))

## Activity

**1.** A classmate suggests skipping `section_instructors` and adding an `instructor_id` column to `sections` instead. Which sections couldn't be stored correctly that way? What else would change about the design?

**2.** Check whether a course (`subject_code` and `course_number`) always has the same `credits`. Which course doesn't, and what does that tell you about where `credits` belongs?

**3.** Our `instructors` table treats two people with the same name as one person, and one person whose name is spelled two ways as two people. Find the last names shared by more than one instructor this term. What would the file need to include to remove this risk?

**4.** Suppose a new instructor named Aaron Abbott joins the faculty and you rerun the pipeline. What happens to the `instructor_id` values? Why is that fine for this pipeline, and when would it become a problem?

**5.** Suppose the registrar sends you a second file next semester, `courses_spring2027.csv`, in the same format. If you ran this whole pipeline on both files, which tables would get new rows? Would any table, key, or step need to change?